In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

base_path = Path().resolve().parent
file_path = base_path / 'data' / 'raw' / 'Stellar_edu_MDS_ap_stats_dataset - v1.9.xlsx'

In [3]:
data = pd.read_excel(file_path, sheet_name='Student_Observations')

In [4]:
data.head()

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q01,MCQ,PCA Q01,KC.U1.02.observational_unit_variable,KC.U1.02.observational_unit_variable,0.0,1,0,C,E,NaN
1,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,100,E,E,NaN
2,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.0,1,0,C,D,NaN
3,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,E,E,NaN
4,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,B,B,NaN


In [5]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 21808 entries, 0 to 21807
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                21808 non-null  str    
 1   assignment_id             21808 non-null  str    
 2   class_num                 21808 non-null  int64  
 3   observation_id            21808 non-null  str    
 4   item_type                 21808 non-null  str    
 5   source_question           21808 non-null  str    
 6   primary_kc_id             21808 non-null  str    
 7   all_kc_ids                21808 non-null  str    
 8   score                     21808 non-null  float64
 9   max_score                 21808 non-null  int64  
 10  percent_score             21808 non-null  int64  
 11  student_response          21808 non-null  str    
 12  correct_answer_or_rubric  21808 non-null  str    
 13  rubric_level              4642 non-null   str    
dtypes: float64(1), in

In [6]:
data.nunique()

student_id                   25
assignment_id                33
class_num                    27
observation_id              895
item_type                     3
source_question             640
primary_kc_id               236
all_kc_ids                  521
score                         3
max_score                     1
percent_score                 3
student_response             25
correct_answer_or_rubric    185
rubric_level                  3
dtype: int64

**The Criteria for the "Best Subset"**
To make BKT work, a Knowledge Component (KC) must meet three strict criteria:

- Vertical Depth (Sequence): Students must have attempted the skill at least 3+ times on average. BKT cannot "see" learning if there is only one data point per person.

- Horizontal Breadth (Reach): At least 5–10 students should have attempted the skill. If only one student did a KC, the model will "overfit" to that one person's specific mistakes.

- Statistical Variance: The KC cannot have 100% accuracy or 0% accuracy. The model needs to see the "flip" from incorrect to correct to calculate the Learn Rate.

In [7]:
data.columns

Index(['student_id', 'assignment_id', 'class_num', 'observation_id',
       'item_type', 'source_question', 'primary_kc_id', 'all_kc_ids', 'score',
       'max_score', 'percent_score', 'student_response',
       'correct_answer_or_rubric', 'rubric_level'],
      dtype='str')

In [ ]:
# check to select KC's that were attempted by at least 3 times per student

check_1 = data.groupby(['primary_kc_id', 'student_id']).size().reset_index(name='attempts')


check_1

,primary_kc_id,student_id,attempts
0,KC.U1.02.observational_unit_variable,S001,2
1,KC.U1.02.observational_unit_variable,S002,2
2,KC.U1.02.observational_unit_variable,S003,2
3,KC.U1.02.observational_unit_variable,S004,2
4,KC.U1.02.observational_unit_variable,S005,2
...,...,...,...
5824,KC.U9.20.regression_inference_scope_and_predic...,S021,2
5825,KC.U9.20.regression_inference_scope_and_predic...,S022,2
5826,KC.U9.20.regression_inference_scope_and_predic...,S023,2
5827,KC.U9.20.regression_inference_scope_and_predic...,S024,2


In [21]:
check_1.attempts.unique()

array([ 2,  3,  4,  7,  6,  1,  5,  8,  9, 14, 13, 12, 17, 15])

In [28]:
check_1.attempts.value_counts(normalize=True) *100


attempts
2     21.770458
3     21.358724
4     13.141191
1     13.021101
5     10.893807
6      8.217533
7      5.781438
8      2.882141
9      1.646938
13     0.428890
14     0.411734
17     0.411734
12     0.017156
15     0.017156
Name: proportion, dtype: float64

In [32]:
greater_3_stud = check_1[check_1['attempts'] >= 3]
greater_3_stud

,primary_kc_id,student_id,attempts
25,KC.U1.03.variable_type_cat_quant,S001,3
26,KC.U1.03.variable_type_cat_quant,S002,3
27,KC.U1.03.variable_type_cat_quant,S003,3
28,KC.U1.03.variable_type_cat_quant,S004,3
29,KC.U1.03.variable_type_cat_quant,S005,3
...,...,...,...
5800,KC.U9.19.lsrl_equation_from_data,S021,3
5801,KC.U9.19.lsrl_equation_from_data,S022,3
5802,KC.U9.19.lsrl_equation_from_data,S023,3
5803,KC.U9.19.lsrl_equation_from_data,S024,3


In [ ]:
# filter to see KCs that were done by a minimum of 5 students

greater_3_stud.groupby('primary_kc_id').filter(lambda x: len(x) >= 5)

,primary_kc_id,student_id,attempts
25,KC.U1.03.variable_type_cat_quant,S001,3
26,KC.U1.03.variable_type_cat_quant,S002,3
27,KC.U1.03.variable_type_cat_quant,S003,3
28,KC.U1.03.variable_type_cat_quant,S004,3
29,KC.U1.03.variable_type_cat_quant,S005,3
...,...,...,...
5800,KC.U9.19.lsrl_equation_from_data,S021,3
5801,KC.U9.19.lsrl_equation_from_data,S022,3
5802,KC.U9.19.lsrl_equation_from_data,S023,3
5803,KC.U9.19.lsrl_equation_from_data,S024,3


In [44]:
greater_3_stud.groupby('primary_kc_id').size().sort_values(ascending=True)

primary_kc_id
KC.U6.10.confidence_level_long_run_interpretation    21
KC.U7.07.ci_width_sample_size_confidence             21
KC.U9.19.lsrl_equation_from_data                     22
KC.U6.08.margin_error_width_sample_size              22
KC.U10.05.bootstrap_percentile_cutoffs               22
                                                     ..
KC.U3.16.explanatory_response_treatments             25
KC.U3.13.stratification_design_advantage             25
KC.U3.11.generalization_scope                        25
KC.U4.11.independent_events_probability              25
KC.U4.32.probability_from_simulation                 25
Length: 157, dtype: int64

In [53]:
# subsetting the data

cleaned_1_data = data[data['primary_kc_id'].isin(greater_3_stud['primary_kc_id'].unique())].reset_index(drop=True)
cleaned_1_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 18679 entries, 0 to 18678
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                18679 non-null  str    
 1   assignment_id             18679 non-null  str    
 2   class_num                 18679 non-null  int64  
 3   observation_id            18679 non-null  str    
 4   item_type                 18679 non-null  str    
 5   source_question           18679 non-null  str    
 6   primary_kc_id             18679 non-null  str    
 7   all_kc_ids                18679 non-null  str    
 8   score                     18679 non-null  float64
 9   max_score                 18679 non-null  int64  
 10  percent_score             18679 non-null  int64  
 11  student_response          18679 non-null  str    
 12  correct_answer_or_rubric  18679 non-null  str    
 13  rubric_level              4046 non-null   str    
dtypes: float64(1), in

Other checks regarding data suitability;

In [54]:
kc_stats = cleaned_1_data.groupby('primary_kc_id').agg(
    total_no_obs=('score', 'count'),
    unique_students=('student_id', 'nunique'),
    avg_score= ('score', 'mean'),
    avg_std=('score', 'std')
)

kc_stats.sort_values(by='total_no_obs')

,total_no_obs,unique_students,avg_score,avg_std
primary_kc_id,,,,
KC.U10.11.cohens_d_interpret_practical_importance,69,23,0.500000,0.469668
KC.U8.21.two_way_p_value_interpretation,69,23,0.420290,0.497222
KC.U7.04.one_sample_t_interval_conditions,69,23,0.405797,0.463961
KC.U6.10.confidence_level_long_run_interpretation,69,25,0.449275,0.501065
KC.U10.07.bootstrap_interval_interpret_context,70,25,0.400000,0.413539
...,...,...,...,...
KC.U4.01.probability_long_run_interpretation,224,25,0.633929,0.482808
KC.U2.03.conditional_relative_frequency,225,25,0.577778,0.483610
KC.U4.11.independent_events_probability,324,25,0.561728,0.496942


In [55]:
kc_stats.describe()

,total_no_obs,unique_students,avg_score,avg_std
count,157.000000,157.000000,157.000000,157.000000
mean,118.974522,24.904459,0.523809,0.478435
std,54.063584,0.371819,0.072336,0.025448
min,69.000000,23.000000,0.340278,0.406748
25%,74.000000,25.000000,0.479885,0.466486
50%,100.000000,25.000000,0.523196,0.484377
75%,148.000000,25.000000,0.565714,0.500000
max,423.000000,25.000000,0.729167,0.503413


In [56]:
min_obs = 30
# min_students = 10
min_var = 0.1

kc_stats[
    (kc_stats['total_no_obs']>=min_obs) &
    #(kc_stats['unique_students']>=min_students) &
    (kc_stats['avg_std']>min_var)
]

,total_no_obs,unique_students,avg_score,avg_std
primary_kc_id,,,,
KC.U1.03.variable_type_cat_quant,75,25,0.666667,0.474579
KC.U1.04.variable_type_discrete_continuous,99,25,0.626263,0.486257
KC.U1.05.categorical_freq_relative,174,25,0.597701,0.491777
KC.U1.06.categorical_bar_pie_segmented,175,25,0.657143,0.476026
KC.U1.07.two_way_table_association,75,25,0.560000,0.499730
...,...,...,...,...
KC.U9.14.slope_test_hypotheses,94,25,0.563830,0.441369
KC.U9.16.slope_test_pvalue_interpret,72,25,0.402778,0.493899
KC.U9.17.slope_test_decision_conclusion,96,25,0.458333,0.473879


In [59]:
cleaned_1_data[cleaned_1_data['primary_kc_id'].isin(kc_stats.index.to_list())]

,student_id,assignment_id,class_num,observation_id,item_type,source_question,primary_kc_id,all_kc_ids,score,max_score,percent_score,student_response,correct_answer_or_rubric,rubric_level
0,S001,HW1,1,HW1_PCA_Q02,MCQ,PCA Q02,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,1.0,1,100,E,E,NaN
1,S001,HW1,1,HW1_PCA_Q03,MCQ,PCA Q03,KC.U1.03.variable_type_cat_quant,KC.U1.03.variable_type_cat_quant,0.0,1,0,C,D,NaN
2,S001,HW1,1,HW1_PCA_Q04,MCQ,PCA Q04,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,E,E,NaN
3,S001,HW1,1,HW1_PCA_Q05,MCQ,PCA Q05,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,B,B,NaN
4,S001,HW1,1,HW1_PCA_Q06,MCQ,PCA Q06,KC.U1.05.categorical_freq_relative,KC.U1.05.categorical_freq_relative,1.0,1,100,D,D,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18674,S025,MF2,27,MF2_FRQ5C,FRQ_Component,FRQ 1,KC.U9.17.slope_test_decision_conclusion,KC.U9.17.slope_test_decision_conclusion|KC.U9....,0.0,1,0,I,Rubric: use test statistic/p-value and conclud...,I
18675,S025,MF2,27,MF2_FRQ6B,FRQ_Component,FRQ 6,KC.U7.12.mean_hypotheses,KC.U7.12.mean_hypotheses|KC.U7.11.matched_pair...,1.0,1,100,E,Rubric: define the population mean paired diff...,E
18676,S025,MF2,27,MF2_FRQ6C,FRQ_Component,FRQ 6,KC.U7.13.t_test_conditions,KC.U7.13.t_test_conditions|KC.U7.11.matched_pa...,0.5,1,50,P,"Rubric: address paired structure, random sampl...",P
18677,S025,MF2,27,MF2_FRQ6D,FRQ_Component,FRQ 6,KC.U7.14.t_test_statistic_mean,KC.U7.14.t_test_statistic_mean|KC.U7.15.p_valu...,1.0,1,100,E,"Rubric: use mean difference, standard error, d...",E


In [ ]:
cleaned_1_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 18679 entries, 0 to 18678
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   student_id                18679 non-null  str    
 1   assignment_id             18679 non-null  str    
 2   class_num                 18679 non-null  int64  
 3   observation_id            18679 non-null  str    
 4   item_type                 18679 non-null  str    
 5   source_question           18679 non-null  str    
 6   primary_kc_id             18679 non-null  str    
 7   all_kc_ids                18679 non-null  str    
 8   score                     18679 non-null  float64
 9   max_score                 18679 non-null  int64  
 10  percent_score             18679 non-null  int64  
 11  student_response          18679 non-null  str    
 12  correct_answer_or_rubric  18679 non-null  str    
 13  rubric_level              4046 non-null   str    
dtypes: float64(1), in

In [60]:
cleaned_1_data.to_csv(base_path / 'data' / 'processed' / 'best_subset_1_data.csv', index=False)

In [63]:
cleaned_1_data.shape

(18679, 14)

In [62]:
data.shape

(21808, 14)